In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

ds=pd.read_csv('/kaggle/input/datasets/lakshyaupadhyaya/movies-cleaned-dataset/movies_cleaned.csv')


def col_for_emotional_analysis(row):
    parts=[]
    if pd.notnull(row['title']):
        parts.append(row['title'])

    if pd.notnull(row['description']):
        parts.append(row['description'])

    return ". ".join(parts)

ds['emotion_analysis']=ds.apply(col_for_emotional_analysis,axis=1)

In [ ]:
ds.columns

In [ ]:
from numpy import result_type
from transformers import pipeline
import pandas as pd
import json
from tqdm import tqdm

tqdm.pandas()
emotion_classifier = pipeline("text-classification", model="samlowe/roberta-base-go_emotions", top_k=None, device=0, truncation=True, max_length=512)
def sentiment_values(row):
    if pd.notnull(row):
        emotions=emotion_classifier(row)[0]
        main_label=emotions[0]['label'] 
        main_score=emotions[0]['score']
        all_emotions=(json.dumps(emotions))
        return pd.Series([main_label,main_score,all_emotions])
    else:
        return pd.Series([None,None,None])

ds[['main_sentiment','main_sentiment_score','all_sentiments']]=ds['emotion_analysis'].progress_apply(
    sentiment_values
)


In [ ]:
ds.to_csv('movie_columns_with_sentiments.csv',index=False)